# Flywheel YEARS Project Workflow

End-to-end notebook covering: querying sessions in a look-back window, converting xlsx files on acquisitions to CSV and re-uploading, reporting on which sessions had xlsx files, checking exported CSVs for empty sheets (with a dry-run delete), and verifying the BIDS export convention on `func-bold` acquisitions.

Appendix at the end has standalone helper snippets (set difference, session-filter debugging).

## Setup

In [1]:
import os
import tempfile

import flywheel
import pandas as pd

fw = flywheel.Client('')
project = fw.projects.find_one('label=YEARS,group=jgruber')
print('project found:', project)

project found: {'analyses': [],
 'created': datetime.datetime(2024, 2, 26, 16, 14, 20, 127000, tzinfo=tzutc()),
 'description': '',
 'editions': {'lab': False},
 'files': [{'classification': {},
            'created': datetime.datetime(2025, 5, 12, 19, 2, 6, 894000, tzinfo=tzutc()),
            'deid_log_id': None,
            'file_id': '66df29860183d5b02bd63f0b',
            'hash': 'c82241a15870ffdb4057146cdd8cc3b848f224a366a75cdf9f3ee2742bcc4085b3ac8a8e448bc1538a7e7a29637c0cb5',
            'id': '8b0abeae-7a0c-4f8e-8d15-55aec995700a',
            'info': {'BIDS': {'Filename': '',
                              'Folder': '',
                              'Path': '',
                              'error_message': "Filename '' should be "
                                               'non-empty',
                              'ignore': False,
                              'rule_id': 'bids_project_file',
                              'template': 'project_file',
                       

## 1. Query sessions in look-back window

Window: `2025-12-01` to `2026-05-31`. Using `<2026-06-01` (rather than `<=2026-05-31`) so sessions created any time during 2026-05-31 are still included, in case `created` carries a time-of-day component.

In [2]:
filtered_sessions = project.sessions.find('created>2025-12-01,created<2026-06-01')
include_list = [s.id for s in filtered_sessions]
print(len(include_list))

44


In [3]:
session = fw.get_session(include_list[0])

In [5]:
session.info["COMPLETENESS"]

{'Analysis ID': '69324c56a45bc143e2bbc548',
 'Any Phase Encoding Discrepancy': False,
 'Duplicates Detected': False,
 'Duplicates List': [],
 'DWI count': 0,
 'Extra Scans List': [],
 'Fieldmap count': 4,
 'Human Eyes': False,
 'Incomplete Acqs': False,
 'Incomplete Acqs List': [],
 'Missing Scans List': [],
 'Phase Encoding Error List': [],
 'Resting state count': 0,
 'Run Downstream Analyses': True,
 'Session Complete': True,
 'Spectroscopy count': 0,
 'Stimulus Complete': True,
 'T1 count': 1,
 'T2 count': 0,
 'Task count': 5}

## 2. Convert xlsx files on acquisitions to CSV and re-upload

Searches every acquisition in the project for `.xlsx`/`.xls` files, converts each sheet to a CSV, and uploads the CSV(s) back onto the same acquisition. Multi-sheet workbooks get one CSV per sheet, suffixed with the sheet name; single-sheet workbooks just get `<name>.csv`. Bad/corrupt excel files are skipped rather than killing the run. Original `.xlsx` files are left in place (not deleted).

In [ ]:
EXCEL_EXTS = ('.xlsx', '.xls')

for session in project.sessions.iter():
    for acquisition in session.acquisitions.iter():
        acquisition = acquisition.reload()  # populate .files
        for f in acquisition.files:
            if not f.name.lower().endswith(EXCEL_EXTS):
                continue

            print(f'Found {f.name} in acquisition {acquisition.label}')

            with tempfile.TemporaryDirectory() as tmpdir:
                local_xlsx = os.path.join(tmpdir, f.name)
                acquisition.download_file(f.name, local_xlsx)

                try:
                    sheets = pd.read_excel(local_xlsx, sheet_name=None)
                except Exception as e:
                    print(f'  skipped, failed to read: {e}')
                    continue

                base_name = os.path.splitext(f.name)[0]
                for sheet_name, df in sheets.items():
                    suffix = '' if len(sheets) == 1 else f'_{sheet_name}'
                    csv_name = f'{base_name}{suffix}.csv'
                    local_csv = os.path.join(tmpdir, csv_name)
                    df.to_csv(local_csv, index=False)

                    acquisition.upload_file(local_csv)
                    print(f'  uploaded {csv_name}')

## 3. Report subjects/sessions that contained xlsx files

Prints `subject`, `session`, and `session_id` for every session with at least one `.xlsx`/`.xls` file, and stores the ids in `xlsx_session_ids` for use downstream.

In [ ]:
xlsx_session_ids = []

for session in project.sessions.iter():
    has_xlsx = False
    for acquisition in session.acquisitions.iter():
        acquisition = acquisition.reload()
        if any(f.name.lower().endswith(('.xlsx', '.xls')) for f in acquisition.files):
            has_xlsx = True
            break

    if has_xlsx:
        print(f'subject={session.subject.label}  session={session.label}  session_id={session.id}')
        xlsx_session_ids.append(session.id)

print(len(xlsx_session_ids))

## 4. Check exported CSVs for empty sheets (dry run first)

Scoped to the sessions in `xlsx_session_ids` (the ones we just converted), since those are the CSVs at risk of having come from an empty excel sheet. Prints every empty CSV's filepath. **`DRY_RUN = True` by default — nothing is deleted until you review the printed list and flip it to `False`.**

In [ ]:
DRY_RUN = True  # flip to False only after reviewing the printed list below

empty_files = []  # (acquisition, filename, filepath) for anything we'd delete

for session in project.sessions.iter():
    if session.id not in xlsx_session_ids:
        continue

    for acquisition in session.acquisitions.iter():
        acquisition = acquisition.reload()
        for f in acquisition.files:
            if not f.name.lower().endswith('.csv'):
                continue

            filepath = f'{session.subject.label}/{session.label}/{acquisition.label}/{f.name}'

            with tempfile.TemporaryDirectory() as tmpdir:
                local_csv = os.path.join(tmpdir, f.name)
                acquisition.download_file(f.name, local_csv)

                try:
                    df = pd.read_csv(local_csv)
                    is_empty = df.empty
                except pd.errors.EmptyDataError:
                    is_empty = True

            if is_empty:
                print(filepath)
                empty_files.append((acquisition, f.name, filepath))

print(f'\n{len(empty_files)} empty csv(s) found.')

if not DRY_RUN:
    for acquisition, fname, filepath in empty_files:
        acquisition.delete_file(fname)
        print(f'deleted: {filepath}')
else:
    print('DRY_RUN is True — nothing deleted. Set DRY_RUN = False to actually delete the files above.')

## 5. Verify BIDS export convention on `func-bold` acquisitions

Scoped to sessions created in the `2025-12-01`–`2026-05-31` look-back window. For every acquisition whose label starts with `func-bold` (excluding `SBRef` acquisitions), confirms there is exactly one BIDS-tagged file whose `info['BIDS']['Filename']` contains `stim` and exactly one whose `Filename` contains `events`. "BIDS-tagged" means `info['BIDS']` is present and is a non-empty dict (the key the Flywheel BIDS curation gear writes). Flags anything that doesn't match.

In [ ]:
def has_bids_metadata(f):
    """True if the file has a non-empty BIDS metadata dict in file.info."""
    bids_info = f.info.get('BIDS') if isinstance(f.info, dict) else None
    return isinstance(bids_info, dict) and bool(bids_info)


flagged = []
checked_count = 0

for session in project.sessions.find('created>2025-12-01,created<=2026-05-31'):
    for acquisition in session.acquisitions.iter():
        if not acquisition.label.startswith('func-bold') or acquisition.label.endswith('SBRef'):
            continue

        acquisition = acquisition.reload()
        checked_count += 1

        recording_files = [
            f for f in acquisition.files
            if has_bids_metadata(f) and 'stim' in f.info['BIDS'].get('Filename', '')
        ]
        events_files = [
            f for f in acquisition.files
            if has_bids_metadata(f) and 'events' in f.info['BIDS'].get('Filename', '')
        ]

        reasons = []
        if len(recording_files) != 1:
            reasons.append(f'{len(recording_files)} BIDS-tagged "recording" file(s) (expected 1)')
        if len(events_files) != 1:
            reasons.append(f'{len(events_files)} BIDS-tagged "events.tsv" file(s) (expected 1)')

        if reasons:
            entry = {
                'subject': session.subject.label,
                'session': session.label,
                'session_id': session.id,
                'acquisition': acquisition.label,
                'acquisition_id': acquisition.id,
                'reasons': reasons,
            }
            flagged.append(entry)
            print(f"FLAGGED: {entry['subject']}/{entry['session']}/{entry['acquisition']} "
                  f"({entry['acquisition_id']}) - {'; '.join(reasons)}")

print(f'\nChecked {checked_count} func-bold acquisition(s), flagged {len(flagged)}.')

---
## Appendix A: keep ids in one list, remove ids found in another

Standalone example: build two id lists from separate queries, then keep everything in the first list except ids that also appear in the second.

In [ ]:
filtered_sessions = project.sessions.find('created>2025-12-01')
list_a = [s.id for s in filtered_sessions]
print(len(list_a))

filtered_sessions = project.sessions.find('created>2026-06-01')
list_b = [s.id for s in filtered_sessions]
print(len(list_b))

exclude = set(list_b)
include_list = [sid for sid in list_a if sid not in exclude]
print(len(include_list))

## Appendix B: debugging an empty session filter

Reference: Flywheel's `find()` syntax joins conditions with a **comma**, not the word `and` (`project.sessions.find('created>2025-12-01,created<2026-06-01')`). If a filtered query still comes back empty, this checklist isolates why: confirm `project` resolved, check the total unfiltered session count, inspect real `created` values, then test each half of the filter separately.

In [ ]:
print('project found:', project)

if project is None:
    raise SystemExit("project.find_one returned None — check label/group spelling/case")

all_sessions = project.sessions()
print('total sessions in project:', len(all_sessions))

for s in all_sessions[:5]:
    print(s.label, s.created)

after = project.sessions.find('created>2025-12-01')
print('sessions after 2025-12-01:', len(after))

before = project.sessions.find('created<2026-06-01')
print('sessions before 2026-06-01:', len(before))

both = project.sessions.find('created>2025-12-01,created<2026-06-01')
print('combined:', len(both))